# 1. Tool calling

You are building the counter concierge for Hearth Lane Café. This notebook is the first piece: one tool, two model calls.

The model must not invent the return rule. It asks for `get_shop_fact`. This process reads `docs/policy.md` and returns the Returns section. The second model call writes the answer from that string.

Later notebooks keep this tool and add more. Run this file from top to bottom. Set `DEMO_MODE` in the next cell if you want to force the scripted model or the live API.

The code cells are the whole agent for this lesson. There is no earlier notebook to open first.

## Setup

`DEMO_MODE = None` follows the environment variable of the same name. If that variable is unset, a missing `OPENAI_API_KEY` selects the scripted demo and a present key selects the Chat Completions API. The key is loaded from the repository-root `.env` by `tutorial/common/client.py`. The value is not printed.

Companion notes: `1-using-tool.md`.

In [ ]:
# None: use the API when OPENAI_API_KEY is set, otherwise the scripted demo.
# True: always the scripted demo. False: always the API (exits if the key is missing).
DEMO_MODE = None

import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import chat, describe_mode, using_demo

LESSON = '1-using-tool'
LESSON_NUMBER = 1
USING_DEMO = using_demo(DEMO_MODE)
print("tutorial:", LESSON)
print(describe_mode(DEMO_MODE))


## Shared helpers

Schema, message copying, and the demo check used by every later cell.

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path

HERE = ROOT / "tutorial"
DOCS = HERE / "docs"
DATA = HERE / "data"
VAR = HERE / "var"

HANDLERS = {}
TOOLS = []


def register(name, description, properties, required, fn):
    """Add one Chat Completions tool. ``fn`` runs only if the harness allows it."""
    spec = {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }
    # Re-running a cell replaces the previous schema instead of stacking a second copy.
    TOOLS[:] = [tool for tool in TOOLS if tool["function"]["name"] != name]
    TOOLS.append(spec)
    HANDLERS[name] = fn


def preview(text, limit=500):
    """Shorten a tool result for the printed trace. The model still gets the full string."""
    text = text if isinstance(text, str) else str(text)
    if len(text) <= limit:
        return text
    return text[:limit] + "\n... [" + str(len(text)) + " characters]"


def assistant_message(turned):
    """Copy an assistant turn into the message list, including any tool calls."""
    message = {"role": "assistant", "content": turned["content"] or ""}
    calls = turned["tool_calls"] or []
    if calls:
        message["tool_calls"] = []
        for call in calls:
            raw = call.get("raw") or json.dumps(call["arguments"], sort_keys=True)
            message["tool_calls"].append({
                "id": call["id"],
                "type": "function",
                "function": {"name": call["name"], "arguments": raw},
            })
    return message


def check(condition, message):
    """Fail the demo when the scripted path drifts. Warn, and continue, on a live model."""
    if condition:
        print("check ok:", message)
        return
    if USING_DEMO:
        raise AssertionError(message)
    print("check warning (live model):", message)


def base_rules():
    return (
        "You are the counter concierge for Hearth Lane Café. "
        "Shop facts come from tools, not from guesses in the prompt. "
        "If a tool does not say, say you don't know. "
        "Cite the tool or the docs path you used. "
        "Do not invent a Wi-Fi password, a refund, or a shipping exception. "
        "Do not charge a card or send email yourself."
    )


def call_tool(call):
    """Run one handler. Bad arguments become an ERROR string, not a crash."""
    name = call["name"]
    args = call["arguments"]
    if not isinstance(args, dict):
        return "ERROR: arguments must be a JSON object."
    if args.get("_error"):
        return "ERROR: " + str(args["_error"])
    fn = HANDLERS.get(name)
    if fn is None:
        return "ERROR: unknown tool " + repr(name) + "."
    return fn(args)


## Tutorial 1 — one tool

`get_shop_fact` opens the shop document. The model only sees the string you return.

In [ ]:
from tutorial.common.tools import read_file


def _read_docs(path):
    """Read one shop document. The path jail lives in ``tutorial.common.tools``."""
    if not isinstance(path, str):
        return "ERROR: path must be a string. Example: policy.md"
    return read_file(str(DOCS), path)


def _section(markdown, heading):
    lines = markdown.splitlines()
    start = None
    prefix = "## " + heading
    for index, line in enumerate(lines):
        if line.strip() == prefix:
            start = index + 1
            break
    if start is None:
        return ""
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    return "\n".join(body).strip()


def get_shop_fact(args):
    """Return one policy or FAQ section. The model names a topic; this function opens the file."""
    topic = str(args.get("topic", "")).strip().lower()
    mapping = {
        "returns": ("policy.md", "Returns"),
        "shipping": ("policy.md", "Shipping"),
        "hours": ("faq.md", "Where and when"),
        "allergens": ("faq.md", "Allergens"),
    }
    if topic not in mapping:
        return "ERROR: topic must be returns, shipping, hours, or allergens."
    path, heading = mapping[topic]
    document = _read_docs(path)
    if document.startswith("ERROR:"):
        return document
    body = _section(document, heading)
    if not body:
        return "ERROR: " + heading + " is missing from " + path + "."
    return "TOPIC: " + topic + "\nSOURCE: docs/" + path + "\n\n" + body


register(
    "get_shop_fact",
    "Look up one Hearth Lane rule. topic is returns, shipping, hours, or allergens. "
    "The result quotes docs/policy.md or docs/faq.md.",
    {"topic": {"type": "string", "description": "returns, shipping, hours, or allergens"}},
    ["topic"],
    get_shop_fact,
)


## System prompt

The prompt stays short. Documents, notes, skills, and pages arrive through tools.

In [ ]:
def system_text():
    parts = [base_rules()]
    parts.append("Use get_shop_fact before you state a return, shipping, hours, or allergen rule.")
    return "\n\n".join(parts)


## Demo

Run this cell after the ones above. In demo mode the checks raise if the trace drifts. A live model prints a warning instead, because the wording can vary.

In [ ]:
print("registered tools:", [tool["function"]["name"] for tool in TOOLS])
print()

question = "Can a customer return an opened bag of house coffee?"
messages = [
    {"role": "system", "content": system_text()},
    {"role": "user", "content": question},
]
print("QUESTION:", question)
print()

# The model proposes a tool call. This process executes it. The model does not open the file.
first = chat(messages, TOOLS, lesson="1-using-tool", demo=DEMO_MODE)
messages.append(assistant_message(first))
names = [call["name"] for call in first["tool_calls"]]
print("tool_calls:", names)
check(names == ["get_shop_fact"] or (names and not USING_DEMO), "demo asks for get_shop_fact")

if not first["tool_calls"]:
    print("ANSWER:", first["content"])
    print("stop: final (no tool call)")
    check(False, "the model asked for a tool before answering")
else:
    for call in first["tool_calls"]:
        result = call_tool(call)
        print("[tool]", call["name"], json.dumps(call["arguments"], sort_keys=True))
        print(preview(result))
        print()
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"],
            "content": result,
        })
    second = chat(messages, TOOLS, lesson="1-using-tool", demo=DEMO_MODE)
    print("ANSWER:", second["content"])
    print("stop: final")
    check(not second["tool_calls"], "the second turn is a final answer")
    check("final sale" in (second["content"] or "").lower(), "answer says opened coffee is final sale")
